# Assignment 2: Data Structures and Methods for HPC
Group 15

- NG, Cheuk Nam (cnng@kth.se)
- Wong, Chun Him (chwon@kth.se)

## Imports

In [207]:
import ipytest
import pytest
import random
import time
import numpy as np
import scipy.stats
from IPython.display import display, Markdown


ipytest.autoconfig()

## Julia Set Code

In [12]:
# area of complex space to investigate
x1, x2, y1, y2 = -1.8, 1.8, -1.8, 1.8
c_real, c_imag = -0.62772, -0.42193

def calc_pure_python(desired_width, max_iterations):
    """Create a list of complex coordinates (zs) and complex parameters (cs),
    build Julia set"""
    x_step = (x2 - x1) / desired_width
    y_step = (y1 - y2) / desired_width
    x = []
    y = []
    ycoord = y2
    while ycoord > y1:
        y.append(ycoord)
        ycoord += y_step
    xcoord = x1
    while xcoord < x2:
        x.append(xcoord)
        xcoord += x_step
    # build a list of coordinates and the initial condition for each cell.
    # Note that our initial condition is a constant and could easily be removed,
    # we use it to simulate a real-world scenario with several inputs to our
    # function
    zs = []
    cs = []
    for ycoord in y:
        for xcoord in x:
            zs.append(complex(xcoord, ycoord))
            cs.append(complex(c_real, c_imag))

    print("Length of x:", len(x))
    print("Total elements:", len(zs))
    start_time = time.time()
    output = calculate_z_serial_purepython(max_iterations, zs, cs)
    end_time = time.time()
    secs = end_time - start_time
    print(calculate_z_serial_purepython.__name__ + " took", secs, "seconds")

    # This sum is expected for a 1000^2 grid with 300 iterations
    # It ensures that our code evolves exactly as we'd intended
    # assert sum(output) == 33219980
    return output

def calculate_z_serial_purepython(maxiter, zs, cs):
    """Calculate output list using Julia update rule"""
    output = [0] * len(zs)
    for i in range(len(zs)):
        n = 0
        z = zs[i]
        c = cs[i]
        while abs(z) < 2 and n < maxiter:
            z = z * z + c
            n += 1
        output[i] = n
    return output

if __name__ == "__main__":
    # Calculate the Julia set using a pure Python solution with
    # reasonable defaults for a laptop
    calc_pure_python(desired_width=1000, max_iterations=300)

calculate_z_serial_purepython took 1.8392739295959473 seconds


# Exercise 1: PyTest with the Julia Set Code

## Task 1.1: Implement a separate code to test the assertion above using the pytest framework
An extra return statement has been added to the `calc_pure_python` function to return the output list. The test function `test_output_sum_for_300_iterations` uses pytest to assert that the sum of the output list for 300 iterations is equal to 33219980.

> The test function below uses the `ipytest` magic command to run pytest within a Jupyter notebook cell. In normal Python files, a `@pytest` decorator would be used instead.

In [23]:
%%ipytest -qq

def test_output_sum_for_300_iterations():
    assert sum(
        calc_pure_python(desired_width=1000, max_iterations=300)
    ) == 33219980

## Task 1.2: How would you implement the unit test with the possibility of having a different number of iterations and grid points?

The above test function in Task 1.1 can be modified to be parameterised as follows:

In [21]:
@pytest.mark.parametrize(
    "width, iterations, expected_sum",
    [
        (1000, 300, 33219980)
    ],
)
def test_output_sum_for_300_iterations_parameterised(width, iterations, expected_sum):
    assert sum(
        calc_pure_python(desired_width=width, max_iterations=iterations)
    ) == expected_sum

# Exercise 2: Python DGEMM Benchmark Operation

## Task 2.1: Implement the DGEMM with matrices as NumPy array

In [94]:
def dgemm_numpy(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> None:
    m, p = a.shape
    n = b.shape[1]

    for i in range(m):
        for j in range(n):
            for k in range(p):
                c[i, j] += a[i, k] * b[k, j]

## Task 2.2: Using pytest develop a unit test for checking the correctness

In [99]:
%%ipytest -qq

def test_dgemm_numpy():
    a = np.array([[11.5, 3.007], [7.0, 11.0]])
    b = np.array([[8.0, 0.0, 1.0], [0.329, 3.0, 5.0]])
    c = np.array([[2.0, 3.0, 9.0], [7.912, 1.0, 2.0]])
    expected_c = np.array([[94.989303, 12.021, 35.535], [67.531, 34.0, 64.0]])
    dgemm_numpy(a, b, c)
    np.testing.assert_array_almost_equal(c, expected_c)

## Task 2.3: Measure the execution time for each approach varying the matrix size


### Code for implementing with Python lists

In [111]:
def dgemm_list(a: list, b: list, c: list) -> None:
    m = len(a)
    p = len(a[0])
    n = len(b[0])

    for i in range(m):
        for j in range(n):
            for k in range(p):
                c[i][j] += a[i][k] * b[k][j]

In [112]:
%%ipytest -qq

def test_dgemm_list():
    a = [[11.5, 3.007], [7.0, 11.0]]
    b = [[8.0, 0.0, 1.0], [0.329, 3.0, 5.0]]
    c = [[2.0, 3.0, 9.0], [7.912, 1.0, 2.0]]
    expected_c = [[94.989303, 12.021, 35.535], [67.531, 34.0, 64.0]]
    dgemm_list(a, b, c)
    np.testing.assert_array_almost_equal(c, expected_c)

### Code for implementing with Python tuples (arrays)

In [114]:
def dgemm_tuple(a: tuple, b: tuple, c: tuple) -> tuple:
    m = len(a)
    p = len(a[0])
    n = len(b[0])

    return tuple(
        tuple(
            c[i][j] + sum(a[i][k] * b[k][j] for k in range(p))
            for j in range(n)
        )
        for i in range(m)
    )

In [115]:
%%ipytest -qq

def test_dgemm_tuple():
    a = ((11.5, 3.007), (7.0, 11.0))
    b = ((8.0, 0.0, 1.0), (0.329, 3.0, 5.0))
    c = ((2.0, 3.0, 9.0), (7.912, 1.0, 2.0))
    expected_c = ((94.989303, 12.021, 35.535), (67.531, 34.0, 64.0))
    new_c = dgemm_tuple(a, b, c)
    np.testing.assert_array_almost_equal(new_c, expected_c)

### Benchmarking

In [192]:
def benchmark_sweep(dgemm_func, init_func, name, start=5, end=100, step=5, iterations=20):
    """
    Runs dgemm_func over a range of matrix sizes and plots the statistics.
    """
    sizes = list(range(start, end + 1, step))
    means, mins, maxs = [], [], []
    cis = [] # Margin of error for CI
    stds = []

    table_lines = [
        f"### Detailed Results: {name}",
        "| Matrix Size | Average (s) | Std Dev (s) | Min (s) | Max (s) | 95% CI Margin (s) |",
        "|:---:|:---:|:---:|:---:|:---:|:---:|"
    ]

    print(f"Running benchmark sweep for {name}...")

    for n in sizes:
        durations = []
        for _ in range(iterations):
            # Initialize matrices using the provided helper
            a, b, c = init_func(n)

            start_time = time.time()
            dgemm_func(a, b, c)
            end_time = time.time()

            durations.append(end_time - start_time)

        # Statistics
        arr = np.array(durations)
        avg = np.mean(arr)
        std = np.std(arr, ddof=1)
        means.append(avg)
        mins.append(np.min(arr))
        maxs.append(np.max(arr))
        stds.append(std)

        # 95% Confidence Interval calculation
        # CI = mean +/- margin_of_error
        # margin_of_error = t * (std / sqrt(n))
        dof = iterations - 1
        t_crit = np.abs(scipy.stats.t.ppf((1 - 0.95) / 2, dof))
        margin_of_error = t_crit * (std / np.sqrt(iterations))
        cis.append(margin_of_error)

        row = f"| {n}x{n} | {avg:.4e} | {std:.4e} | {np.min(arr):.4e} | {np.max(arr):.4e} | ±{margin_of_error:.4e} |"
        table_lines.append(row)

    display(Markdown("\n".join(table_lines)))

#### Benchmarking for Python lists

##### Initialiser

In [172]:
def init_list_matrices(n):
    a = [[random.random() for _ in range(n)] for _ in range(n)]
    b = [[random.random() for _ in range(n)] for _ in range(n)]
    c = [[0.0 for _ in range(n)] for _ in range(n)]
    return a, b, c

##### Benchmark Execution

In [204]:
benchmark_sweep(dgemm_list, init_list_matrices, "Python Lists", start=10, end=100, step=10, iterations=50)

### Detailed Results: Python Lists
| Matrix Size | Average (s) | Std Dev (s) | Min (s) | Max (s) | 95% CI Margin (s) |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 10x10 | 8.1687e-05 | 9.9661e-06 | 7.1049e-05 | 1.2088e-04 | ±2.8323e-06 |
| 20x20 | 4.8196e-04 | 6.7472e-05 | 3.9196e-04 | 6.8784e-04 | ±1.9175e-05 |
| 30x30 | 1.1182e-03 | 9.5056e-05 | 9.7013e-04 | 1.3838e-03 | ±2.7015e-05 |
| 40x40 | 2.4643e-03 | 9.3585e-05 | 2.2547e-03 | 2.7499e-03 | ±2.6596e-05 |
| 50x50 | 4.9397e-03 | 1.8874e-04 | 4.5300e-03 | 5.4290e-03 | ±5.3638e-05 |
| 60x60 | 8.3588e-03 | 1.2205e-04 | 8.0299e-03 | 8.7471e-03 | ±3.4685e-05 |
| 70x70 | 1.3137e-02 | 2.1648e-04 | 1.2641e-02 | 1.3634e-02 | ±6.1523e-05 |
| 80x80 | 1.9637e-02 | 2.5855e-04 | 1.8815e-02 | 2.0101e-02 | ±7.3479e-05 |
| 90x90 | 2.8377e-02 | 6.5097e-04 | 2.7314e-02 | 2.9595e-02 | ±1.8500e-04 |
| 100x100 | 3.8517e-02 | 5.4605e-04 | 3.7727e-02 | 4.0397e-02 | ±1.5519e-04 |

#### Benchmarking for Python tuples

##### Initialiser

In [194]:
def init_tuple_matrices(n):
    """Initializes matrices as tuples for the requested size n."""
    a = tuple(tuple(random.random() for _ in range(n)) for _ in range(n))
    b = tuple(tuple(random.random() for _ in range(n)) for _ in range(n))
    c = tuple(tuple(0.0 for _ in range(n)) for _ in range(n))
    return a, b, c

##### Benchmark Execution

In [203]:
benchmark_sweep(dgemm_tuple, init_tuple_matrices, "Python Tuples", start=10, end=100, step=10, iterations=50)

### Detailed Results: Python Tuples
| Matrix Size | Average (s) | Std Dev (s) | Min (s) | Max (s) | 95% CI Margin (s) |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 10x10 | 1.0435e-04 | 2.6962e-05 | 9.1791e-05 | 2.4700e-04 | ±7.6626e-06 |
| 20x20 | 4.8028e-04 | 6.3528e-05 | 3.9411e-04 | 6.8784e-04 | ±1.8055e-05 |
| 30x30 | 1.0978e-03 | 7.6868e-05 | 9.7513e-04 | 1.2569e-03 | ±2.1846e-05 |
| 40x40 | 2.4012e-03 | 8.4583e-05 | 2.2280e-03 | 2.7461e-03 | ±2.4038e-05 |
| 50x50 | 4.6327e-03 | 1.3172e-04 | 4.3452e-03 | 4.8969e-03 | ±3.7436e-05 |
| 60x60 | 7.9803e-03 | 2.2889e-04 | 7.5471e-03 | 8.4639e-03 | ±6.5048e-05 |
| 70x70 | 1.2452e-02 | 3.2135e-04 | 1.2045e-02 | 1.3240e-02 | ±9.1326e-05 |
| 80x80 | 1.8023e-02 | 2.8103e-04 | 1.7452e-02 | 1.8982e-02 | ±7.9867e-05 |
| 90x90 | 2.5635e-02 | 5.2794e-04 | 2.5105e-02 | 2.7178e-02 | ±1.5004e-04 |
| 100x100 | 3.4453e-02 | 1.7509e-04 | 3.4091e-02 | 3.5347e-02 | ±4.9760e-05 |

#### Benchmarking for NumPy arrays

##### Initialiser

In [189]:
def init_numpy_matrices(n):
    """Initializes matrices as NumPy arrays for the requested size n."""
    a = np.random.rand(n, n)
    b = np.random.rand(n, n)
    c = np.zeros((n, n))
    return a, b, c

##### Benchmark Execution

In [201]:
benchmark_sweep(dgemm_numpy, init_numpy_matrices, "NumPy Arrays", start=10, end=100, step=10, iterations=50)

### Detailed Results: NumPy Arrays
| Matrix Size | Average (s) | Std Dev (s) | Min (s) | Max (s) | 95% CI Margin (s) |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 10x10 | 2.9938e-04 | 3.4639e-05 | 2.5010e-04 | 3.8910e-04 | ±9.8442e-06 |
| 20x20 | 1.6458e-03 | 1.5185e-04 | 1.5182e-03 | 2.1276e-03 | ±4.3156e-05 |
| 30x30 | 5.2404e-03 | 1.4341e-04 | 5.0781e-03 | 5.7080e-03 | ±4.0755e-05 |
| 40x40 | 1.2726e-02 | 3.7605e-04 | 1.2066e-02 | 1.4115e-02 | ±1.0687e-04 |
| 50x50 | 2.4407e-02 | 6.7536e-04 | 2.3508e-02 | 2.5853e-02 | ±1.9194e-04 |
| 60x60 | 4.3653e-02 | 5.2564e-03 | 4.0631e-02 | 6.2964e-02 | ±1.4939e-03 |
| 70x70 | 7.9405e-02 | 2.8232e-02 | 6.4548e-02 | 2.0552e-01 | ±8.0235e-03 |
| 80x80 | 9.7417e-02 | 6.7181e-04 | 9.5891e-02 | 9.9707e-02 | ±1.9093e-04 |
| 90x90 | 1.3842e-01 | 9.3904e-04 | 1.3693e-01 | 1.4129e-01 | ±2.6687e-04 |
| 100x100 | 1.9191e-01 | 2.5164e-03 | 1.8833e-01 | 2.0141e-01 | ±7.1514e-04 |

#### Benchmark Results Analysis

The results indicate that Python lists and tuples yield comparable average computation times. Contrary to common expectation, NumPy arrays do not outperform either structure, despite their optimised C implementation. This arises because the nested loops in the DGEMM implementation bypass NumPy's vectorised operations, resulting in performance akin to that of native Python data structures. Indeed, direct element access in NumPy arrays proves slower than in lists or tuples, with computation times approximately one order of magnitude higher. This might be attributed to the overhead of NumPy's internal handling of array accesses, having to create temporary Python `float` objects for each element access.

As matrix dimensions increase, both the mean computation time and the standard deviation rise. The growth in mean time is expected, given the O(n³) complexity of the algorithm. The increase in standard deviation and consequently the widening of the confidence interval likely stems from greater susceptibility to external interference during longer executions. Such interference may include additional context switches, scheduling delays, or other system-level activity. For larger matrices, cache misses and increased main-memory access would further contribute to execution-time variability.

## Task 2.4: Using the timing information and the number of operations for the DGEMM, calculate the FLOPS/s

In each of the DGEMM implementations, the total number of floating-point operations (FLOPs) for multiplying two n x n matrices is given by $2n^3$ as each loop iteration contains one floating-point multiplication and one floating-point addition.

Taking the average execution time of a 100 x 100 matrix multiplication using Python Lists as an example, which is approximately 0.038517 seconds, we can calculate the FLOPS/s as follows:

$$
\text{FLOPS/s} = \frac{2(100)^3}{0.038517} \approx 51925124 \text{ FLOPS/s}
$$

All the results for this task are done on Apple M4 chip. Assuming that one operation is done per cycle, then the theoretical peak is the clock frequency value: 4.3 GHz or 4.3 x 10^9 FLOPS/s.

This implies that the achieved performance of approximately 51.9 MFLOPS/s is about 1.2% of the theoretical peak performance of the Apple M4 chip and one floating point operation might not be done per cycle.

## Task 2.5: Compare the performance results with the numpy matmul operation

In [53]:
def dgemm_numpy_matmul(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> None:
    c += np.matmul(a, b)

### Benchmark Execution

In [202]:
benchmark_sweep(dgemm_numpy_matmul, init_numpy_matrices, "NumPy Arrays with matmul", start=10, end=100, step=10, iterations=50)

### Detailed Results: NumPy Arrays with matmul
| Matrix Size | Average (s) | Std Dev (s) | Min (s) | Max (s) | 95% CI Margin (s) |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 10x10 | 2.9278e-06 | 6.3969e-06 | 7.1526e-07 | 3.2187e-05 | ±1.8180e-06 |
| 20x20 | 3.3760e-06 | 6.5312e-06 | 9.5367e-07 | 4.2915e-05 | ±1.8561e-06 |
| 30x30 | 2.7084e-06 | 3.4933e-06 | 9.5367e-07 | 2.6226e-05 | ±9.9279e-07 |
| 40x40 | 3.1185e-06 | 2.5442e-06 | 1.6689e-06 | 1.5974e-05 | ±7.2305e-07 |
| 50x50 | 4.7922e-06 | 5.0655e-06 | 2.8610e-06 | 3.1948e-05 | ±1.4396e-06 |
| 60x60 | 4.2772e-06 | 6.0433e-07 | 3.8147e-06 | 7.1526e-06 | ±1.7175e-07 |
| 70x70 | 6.1131e-06 | 7.7183e-07 | 4.7684e-06 | 9.0599e-06 | ±2.1935e-07 |
| 80x80 | 6.8426e-06 | 8.1352e-07 | 5.7220e-06 | 1.0967e-05 | ±2.3120e-07 |
| 90x90 | 9.5367e-06 | 2.0345e-06 | 8.8215e-06 | 2.3127e-05 | ±5.7819e-07 |
| 100x100 | 1.1983e-05 | 1.3425e-06 | 1.0967e-05 | 1.8120e-05 | ±3.8153e-07 |

## Benchmark Results Analysis for NumPy matmul

The NumPy `matmul` function significantly outperforms the custom DGEMM implementation using NumPy arrays, tuples and lists. This is expected, as `matmul` leverages optimised low-level libraries (BLAS) that are vectorised. The performance difference becomes more evident with increasing matrix sizes, despite the runtime for smaller matrices remain similar. It can also be observed that the standard deviation and confidence intervals for `matmul` remains relatively constant across matrix sizes, indicating consistent performance likely due to efficient memory access patterns and optimised algorithms.

# Exercise 3: Experiment with the Python Debugger

## Task 3.1: What are the advantages of using a debugger? What challenges did you find in using the pdb debugger, if any?

One of the advantages of using a debugger is that one can inspect the state of variables at any point during execution, eliminating the need for programmers to rely on print statements. Additionally, it enables programmers to step through the code line by line, placing the flow of execution under a direct control.

Although this is one of my first experiences using the Python debugger, I found it very easy to use, given my prior experience with LLDB and GDB. It also provides useful pretty-printing functionality. However, compared to LLDB, the debugger appears less feature-rich, lacking capabilities such as reverse stepping. Overall, I encountered no significant challenges when using pdb.

# Bonus Exercise: Performance Analysis and Optimization of the Game of Life Code